<a href="https://colab.research.google.com/github/osoria80/03MIAR-Algoritmos-de-optimizacion/blob/main/Proyecto/Proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos:   <br>
Url: https://github.com/osoria80/03MIAR-Algoritmos-de-optimizacion/tree/main/Proyecto<br>
Problema:
> 1. Sesiones de doblaje <br>

Descripción del problema:
Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las
tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de
grabación independientemente del número de tomas que se graben. No es posible grabar más
de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los
servicios de los actores de doblaje sea el menor posible.

(*) La respuesta es obligatoria





                                        

# (*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




**Sin restricciones:**

Para calcular el total de posibilidades de ordenación de las tomas, debemos calcular sus posibles permitaciones, sin posibilidad de repetir tomas, que viene dado por la fórmula $P = n!$. Teniendo en cuenta que nuestro número de tomas és 30 (n = 30):

$$P = 30! = 265252859812191058636308480000000$$


In [ ]:
import math
p = math.factorial(30)
print(p)

265252859812191058636308480000000


# Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, argumentalo)


Respuesta<br>
**Primera opción: Listas anidadas**


Para representar la solucion, optamos por utilizar una lista de lista, donde cada sublista representa un día de grabación, y cada entero dentro de cada sublista es el índice de una toma. Puede haber tantas listas como días posibles (máximo 30) y dentro de cada sublista entre 1 y 6 tomas.

$$Solución = [[t_{1,1},t_{1,2},..,t_{1,h}],[t_{2,1},..,t_{2,i}],..,[t_{k,1},..,t_{k,j}]]$$

Dónde: $0 < h,i,j \leqslant 6, 5 \leqslant k \leqslant 30$

Beneficios de esta estructura:


*   Sencilla de interpretar: Tantos días como listas, y tantas tomas como elementos dentro de la sublista. Interpretable a simple vista.
*   Fácil acceso a actores por día: Identificar la toma por su índice, nos permite un cálculo muy sencillo y eficiente del coste por día, siendo este una operación 'OR' de todas las máscaras con indice en la sublista.
* Cálculo de coste total: siguienod el punto anterior, el coste total será la suma de los costes de la lista principal.
* Operabilidad: Acceso sencillo por indexación y operabilidad optimizada con listas.


**Segunda opción: Una sola lista de 30 elementos**


Tras diseñar la estrategia, nos damos cuenta que podemos fijar que cada día (menos el último) se hagan con el máximo de sesiones (6). Por lo que para identificar los días, no hace falta separar en sublistas, sino que los indices nos indican a qué día pertenecen.

$$Solución=[t_1,t_2,..,t_k]$$
Dónde: $0 \leqslant k \leqslant 30$


In [ ]:
#Inicializamos una solución vacía de 5 días (mínimo de días: 30 sesiones / 6 por día = 5)
#solucion = [0]*len(df)

# Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?


(*)¿Es un problema de maximización o minimización?

Respuesta<br>
Siendo $X_k$ el número de actores presentes en cada día. La función objeivo será el mínimo de la suma total de actores de todos los días.

El coste diario $X_k$, se calculará como una codificación binária de 10 bits, dónde cada bit representa a un actor.

$Bit_n=FALSE \rightarrow \text{Actor n ausente}$<BR>
$Bit_n=TRUE \rightarrow \text{Actor n presente}$
$$X_k:\{1,..,10\}\rightarrow\{1,0\}$$

Por lo tanto, nuestra función objetivo será:

$$Objetivo = Mín X_1 + X_2 + .. + X_k$$


Es un problema de minimización. Queremos que el número de actores presentes cada día sea el mínimo posible, de esta forma, el coste diário será también el mínimo, y consigo el coste total.

In [1]:
#Definimos una función para calcular el coste del día
def coste_dia(tomas):
  mask_dia = 0
  for toma in tomas:
    mask_toma = int(''.join(df.loc[str(toma)].astype(int).astype(str)), 2)
    mask_dia |= mask_toma
  #print(f"Coste del día: {bin(mask_dia).count('1')}")
  return bin(mask_dia).count('1')

#Definimos una función para calcular el coste total
def coste_total(solucion):
  coste = 0
  dias = (len(solucion) + 6 - 1) // 6
  #print(f"Días de rodaje: {dias}")
  for dia in range(dias):
    if dia == dias: coste += coste_dia(solucion[dia*6:])
    else: coste += coste_dia(solucion[dia*6:(dia+1)*6])
  #print(f"Coste total: {coste}")
  return coste



# Diseña un algoritmo para resolver el problema por fuerza bruta

Antes de resolverlo, fijamos las restricciones y definimos una estrategia.

**Restricciones:**
* Máximo 6 tomas por día
* Que se hagan todas las tomas

**Estrategia:**

Definimos que la estrategia a seguir será la de hacer el máximo de tomas pemritidas por día. De esta manera reducimos el nº de días, y por tanto el coste total. Por ello, podemos asumir que todos los días tendrán 6 tomas, exceptuando el último día, el cual se grabarán el restante de las tomas.

**Fuerza bruta:**

Generaremos todas las permutaciones posibles de las 30 sesiones para calcular su coste. Nos quedaremos con la permutación con el menor coste.

Para poder comprobar el código, trabajaremos con una versiones reducida del probelma (5 tomas y 3 actores, o 7 tomas y 10 actores, o 10 tomas y 10 actores)


In [40]:
#Realizamos la carga de datos
import pandas as pd
df_ini = pd.read_csv('Datos problema doblaje(30 tomas, 10 actores).csv', index_col=0, header=1).iloc[:30,:10]
df_ini = df_ini.astype(int)

#Utilizamos un dataset más pequeño para hacer comprobaciones
df=df_ini.iloc[:9,:10]
display(df)

,1,2,3,4,5,6,7,8,9,10
Toma,,,,,,,,,,
1,1,1,1,1,1,0,0,0,0,0
2,0,0,1,1,1,0,0,0,0,0
3,0,1,0,0,1,0,1,0,0,0
4,1,1,0,0,0,0,1,1,0,0
5,0,1,0,1,0,0,0,1,0,0
6,1,1,0,1,1,0,0,0,0,0
7,1,1,0,1,1,0,0,0,0,0
8,1,1,0,0,0,1,0,0,0,0
9,1,1,0,1,0,0,0,0,0,0


In [41]:
#Generamos la primera solución -> Sesiones ordenadas
solucion_ini = [i for i in range(1, len(df)+1)]
coste_min = float('inf') #Inicializamos el valor del coste mínimo
solucion_min = [] #Inicializamos el resultado mínimo

#Definimos la función para generar todas las permutaciones
def permutaciones(solucion, inicio=0):
  global coste_min, solucion_min
  #Comprobamos si esa permutación es la de mínimo coste
  if inicio == len(solucion):
    c = coste_total(solucion)
    if c < coste_min:
      coste_min = c
      solucion_min = solucion[:]
    return

  for i in range(inicio, len(solucion)):
    # Ponemos solucion[i] en la posición 'inicio'
    solucion[inicio], solucion[i] = solucion[i], solucion[inicio]
    #Llamamos a 'permutaciones' para permutar el resto de valores
    permutaciones(solucion, inicio + 1)
    # Deshacemos el swap (backtrack) para realizar la siguiente permutación
    solucion[inicio], solucion[i] = solucion[i], solucion[inicio]

  return solucion

permutaciones(solucion_ini)
print(f"Mejor solución: {solucion_min}\nCoste: {coste_min}")



Mejor solución: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Coste: 12


# Calcula la complejidad del algoritmo por fuerza bruta

Primero analizamos el coste de la función recursiva 'permutaciones':


*   Para cada solución posible, la función se va a llamar n veces. Para generar los n elementos de la solución.
*   Para calcular todas las permutaciones, la función se va a llamar n! veces. Para generar todas las permutaciones posibles.
*   El resto de operaciones tienen un coste constantes.

El coste computacional de 'permutaciones' es $O(n!)$.


Ahora analizamos el coste de la función 'coste_total':


*   Para cada llamada, se recorre la lista de n elementos 1 vez.
*   El resto de operaciones tienen un coste constante.

Teniendo en cuenta que la función 'coste_total' se llama en cada permutación completa, obtenemos el coste computacional total:

$$\text{Coste computacional: }O(n!·n)$$
<br>
Es un coste extremadamente elevado. Solo para 10 tomas, ya tenemos aproximadamente 36 millones de operaciones. Solo con aumentar 5 tomas más (15 tomas), aumentamos a 13 billones de operaciones. Por lo que para dimensiones mayores, como nuestro problema inicial (30 tomas), tendríamos $2.65\times10^{33}$ operaciones, algo inviable de computar.




# (*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Para solucionar un problema de este tipo, un algoritmo de ramificación y poda (Branch & Bound) sería la mejor opción.
En esencia, sería el mismo algoritmo de fuerza bruta, pero añadiendo una cota inferior que permite podar ramas enteras sin explorarlas.

Definiremos nuestro criterio de poda como el siguiente:


$$\text{Coste parcial (días ya completados) + Cota inferior (días restantes)} \geq \text{coste_min → Podar: esta rama nunca mejorará la solución actual}$$

Para tener un algoritmo que mejore de manera notable la ocomplejidad compiutaciones, debemos definir una buena cota inferior, que sea rápida de calcular y intentar no sobreestimar el coste real de la rama.

Definimos nuetsra cota inferior:


*   En el mejor caso posible, las tomas restantes de ordenar se agruparían perfectamente para minimizar actores (solución objetivo).
*   La cota asume el escenario más optimista posible:
* En cada día futuro, el número de actores es el mínimo de las tomas restantes.
* Eso quiere decir, que en el mejor de los caso, el coste del día+1 será el de la toma con menos actores de las tomas restantes.

De esta manera no sobreestimamos las ramas ya que estamos siendo muy optimistas con las soluciones futuras. Así, la cota generalmente será más pequeña o igual al coste real.






In [42]:
#Realizamos la carga de datos
import pandas as pd
df_ini = pd.read_csv('Datos problema doblaje(30 tomas, 10 actores).csv', index_col=0, header=1).iloc[:30,:10]
df_ini = df_ini.astype(int)

#Utilizamos un dataset más pequeño para hacer comprobaciones
df=df_ini.iloc[:20,:10]
display(df)

,1,2,3,4,5,6,7,8,9,10
Toma,,,,,,,,,,
1,1,1,1,1,1,0,0,0,0,0
2,0,0,1,1,1,0,0,0,0,0
3,0,1,0,0,1,0,1,0,0,0
4,1,1,0,0,0,0,1,1,0,0
5,0,1,0,1,0,0,0,1,0,0
6,1,1,0,1,1,0,0,0,0,0
7,1,1,0,1,1,0,0,0,0,0
8,1,1,0,0,0,1,0,0,0,0
9,1,1,0,1,0,0,0,0,0,0


In [29]:
#Definimos una función para calcular la cota inferior
def cota_inferior (solucion, inicio):
  #Calculamos el coste de los días ya completados
  if inicio % 6 != 0:
    coste_parcial = coste_total(solucion[:inicio - (inicio % 6)])
  else:
    coste_parcial = coste_total(solucion[:inicio])

  #Calculamos el coste de las tomas del día actual
  if inicio % 6 != 0:
    tomas_dia_actual = solucion[inicio - (inicio % 6):inicio]
    coste_dia_actual = coste_dia(tomas_dia_actual)
  else:
    coste_dia_actual = 0

  #Generamos una lista con las tomas restantes
  tomas_restantes = solucion[inicio:]
  #Ordenamos esa lista de menor a mayor coste para elegir la de menor coste
  costes_tomas = []
  for t in tomas_restantes:
    mask_toma = int(''.join(df.loc[str(t)].astype(int).astype(str)), 2)
    costes_tomas.append(bin(mask_toma).count('1'))
  costes_tomas.sort()
  #Calculamos el coste más optimista de los siguientes días
  coste_futuro = 0
  for i in range(0, len(costes_tomas), 6):
    coste_futuro += costes_tomas[i]  # mínimo del grupo = el primero (ordenados asc)
  coste_rama = coste_parcial + coste_dia_actual + coste_futuro
  return coste_rama


In [30]:
#Definimos el algoritmo de branch and bound
def branch_and_bound(solucion, inicio=0):
  global coste_min, solucion_min

  #Realizamos la poda si la cota inferior de la rama es mayor al coste mínimo obtenido
  if cota_inferior(solucion, inicio) >= coste_min:
    return
  #A partir de aquí utilizamos el algoritmo de fuerza bruta para generar las las permutaciones
  if inicio == len(solucion):
    c = coste_total(solucion)
    if c < coste_min:
      coste_min = c
      solucion_min = solucion[:]
    return

  for i in range(inicio, len(solucion)):
    solucion[inicio], solucion[i] = solucion[i], solucion[inicio]
    branch_and_bound(solucion, inicio + 1)
    solucion[inicio], solucion[i] = solucion[i], solucion[inicio]

In [43]:
#Inicializamos la solución optima
coste_min = float('inf') #Inicializamos el valor del coste mínimo
solucion_min = [] #Inicializamos el resultado mínimo

#Ejecutamos el algoritmo
solucion_ini = [i for i in range(1, len(df)+1)]
branch_and_bound(solucion_ini)
print(f"Mejor solución: {solucion_min}\nCoste: {coste_min}")

KeyboardInterrupt: 

# (*)Calcula la complejidad del algoritmo

Respuesta

# Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

# Aplica el algoritmo al juego de datos generado

Respuesta

# Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

# Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta